In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Quantum Neural Network for Breast Cancer Classification

## Objective
Build and simulate a hybrid Quantum Neural Network (QNN)
for binary classification of breast cancer samples.

## Dataset
Breast Cancer Wisconsin (Diagnostic)

## Framework
PennyLane

## Quantum Architecture
- 4 qubits
- Angle encoding using RY rotations
- Trainable variational rotations
- CNOT entangling gates
- Pauli-Z expectation value measurement

## Classification
Benign vs Malignant

In [2]:
!pip install pennylane

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 39.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 103.3 MB/s  0:00:00
  Attempting uninstall: autograd╺━━━━━━━━━━━━━━━━━━━━━━━━ 3/8 [diastatic-malt]2]
    Found existing installation: autograd 1.9.1━━━━━━━━━━━━━━━ 3/8 [diastatic-malt]
    Uninstalling autograd-1.9.1:━╺━━━━━━━━━━━━━━ 5/8 [autograd]t]
      Successfully uninstalled autograd-1.9.190m━━━━━━━━━━━━━━ 5/8 [autograd]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]/8 [pennylane]lightning]


In [3]:
import pennylane as qml

dev = qml.device("default.qubit", wires=4)

print("PennyLane works!")

PennyLane works!


In [4]:
import pennylane as qml
from pennylane import numpy as np

import matplotlib.pyplot as plt
import pandas as pd

print("PennyLane version:", qml.__version__)

PennyLane version: 0.45.1


In [5]:
N_QUBITS = 4

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

print("Quantum device initialized successfully.")
print("Number of qubits:", N_QUBITS)
#we Created a simulated quantum device with four qubits. This is where our circuit will run.

Quantum device initialized successfully.
Number of qubits: 4


In [7]:
@qml.qnode(dev)
def qnn_circuit(inputs, weights):

    # 1. Encode the input features
    for wire in range(N_QUBITS):
        qml.RY(inputs[wire], wires=wire)

    # 2. Apply trainable rotations
    for wire in range(N_QUBITS):
        qml.RY(weights[0, wire], wires=wire)

    # 3. Entangle neighboring qubits
    for wire in range(N_QUBITS - 1):
        qml.CNOT(wires=[wire, wire + 1])

    # 4. Measure the first qubit
    return qml.expval(qml.PauliZ(0))

In [8]:
sample_input = np.array(
    [0.3, 0.7, 0.2, 0.9]
)

initial_weights = np.array(
    [[0.1, 0.1, 0.1, 0.1]],
    requires_grad=True
)

output = qnn_circuit(sample_input, initial_weights)

print("Circuit output:", output)
#verifying the cercuit works

Circuit output: 0.9210609940028852


In [9]:
print(
    qml.draw(qnn_circuit)(
        sample_input,
        initial_weights
    )
)

0: ──RY(0.30)──RY(0.10)─╭●───────┤  <Z>
1: ──RY(0.70)──RY(0.10)─╰X─╭●────┤     
2: ──RY(0.20)──RY(0.10)────╰X─╭●─┤     
3: ──RY(0.90)──RY(0.10)───────╰X─┤     


In [10]:
gradient = qml.grad(
    lambda weights: qnn_circuit(sample_input, weights)
)(initial_weights)

print("Gradients:", gradient)
#Checkin that the circuit can learn ''calc gradient

Gradients: [[-3.89418342e-01  1.17961196e-16 -2.08166817e-17 -4.16333634e-17]]
